# Customer Churn Prediction – Model Optimization
### Academic Mini Project
This notebook details the systematic development and optimization of machine learning classifiers to predict customer churn in a telecom firm. We evaluate models, conduct hyperparameter search using nested pipelines, and implement a custom gradient descent solver from scratch in NumPy.

## Step 1: Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, auc

# Set visualization styles
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

## Step 2: Load and Inspect Dataset
We load the dataset from the local folder. If it is missing, we download it from our mirror.

In [ ]:
from src.data_preprocessing import load_data

raw_df = load_data()
print(f"Dataset shape: {raw_df.shape}")
raw_df.info()

## Step 3: Data Cleaning
We address blank spaces in `TotalCharges` (coerce to NaN and fill with `0.0` for new customers where tenure is 0) and encode the target variable `Churn` to `0` and `1`.

In [ ]:
from src.data_preprocessing import clean_data

cleaned_df = clean_data(raw_df)
print(f"Cleaned dataset shape: {cleaned_df.shape}")
cleaned_df.head()

## Step 4: Exploratory Data Analysis (EDA)
We create visualizations showing churn distribution and correlations. These are saved in `outputs/figures/`.

In [ ]:
from src.eda import generate_all_eda_plots

generate_all_eda_plots(cleaned_df)

## Step 5: Stratified Train-Test Split
We split features and target, using stratify to maintain class distributions.

In [ ]:
from src.data_preprocessing import split_data

X_train, X_test, y_train, y_test = split_data(cleaned_df)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
print("Train Churn Rate:\n", y_train.value_counts(normalize=True))

## Step 6: Preprocessing Configuration
We construct our `ColumnTransformer` with zero data leakage: categorical features are One-Hot Encoded and numerical features are Standard Scaled. Note: fitting will be done inside Pipeline objects during model training.

In [ ]:
from src.data_preprocessing import get_preprocessor

preprocessor = get_preprocessor()
print(preprocessor)

## Step 7: Baseline Models & Evaluation
We initialize baseline classifiers and evaluate them using Stratified 5-Fold Cross Validation on the training data.

In [ ]:
from src.models import get_baseline_models
from src.optimization import run_baseline_cross_validation

baseline_models = get_baseline_models()
cv_results = run_baseline_cross_validation(X_train, y_train, preprocessor, baseline_models)

## Step 8: GridSearchCV Hyperparameter Tuning
To maximize F1-score, we wrap preprocessing and estimators into an sklearn `Pipeline` and run Grid Search CV. This guarantees that preprocessing coefficients are fit only on training folds, avoiding data leakage.

In [ ]:
from src.optimization import perform_grid_search

tuned_pipelines, tuning_df = perform_grid_search(X_train, y_train, preprocessor, baseline_models)

## Step 9: Before vs After Optimization Comparison
We evaluate baseline models and optimized models on the test set and compare the changes in metrics.

In [ ]:
from src.optimization import evaluate_and_compare_models

results_df = evaluate_and_compare_models(
    baseline_models, tuned_pipelines, 
    X_train, y_train, X_test, y_test, preprocessor
)

## Step 10: Best Model Selection and Serialization
We programmatically select the pipeline with the highest test set F1-Score and serialize it as a single file.

In [ ]:
from src.optimization import select_and_save_best_model

best_name, best_pipe = select_and_save_best_model(tuned_pipelines, results_df)

## Step 11: Custom Gradient Descent from Scratch
We train a custom Logistic Regression model from scratch in NumPy using Gradient Descent to study convergence.

In [ ]:
from src.gradient_descent import run_learning_rate_experiments, run_gd_vs_sklearn_comparison

# Fit preprocessor separately on training data to scale input for GD solver
preprocessor.fit(X_train, y_train)
X_train_scaled = preprocessor.transform(X_train)
X_test_scaled = preprocessor.transform(X_test)

# 1. Run learning rate experiments
lr_experiments = run_learning_rate_experiments(X_train_scaled, y_train, X_test_scaled, y_test)

# 2. Run head-to-head comparison
gd_vs_sklearn = run_gd_vs_sklearn_comparison(X_train_scaled, y_train, X_test_scaled, y_test)

## Step 12: Feature Importance Analysis
We visualize the top factors contributing to customer churn.

In [ ]:
classifier = best_pipe.named_steps['classifier']
feature_names = best_pipe.named_steps['preprocessor'].get_feature_names_out()
feature_names = [f.replace('num__', '').replace('cat__', '') for f in feature_names]

if hasattr(classifier, 'feature_importances_'):
    importances = classifier.feature_importances_
    fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    fi_df = fi_df.sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=fi_df.head(10), x='Importance', y='Feature', palette='viridis')
    plt.title('Top 10 Feature Importances')
    plt.show()
elif hasattr(classifier, 'coef_'):
    coefs = np.abs(classifier.coef_[0])
    fi_df = pd.DataFrame({'Feature': feature_names, 'Coefficient Magnitude': coefs})
    fi_df = fi_df.sort_values(by='Coefficient Magnitude', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=fi_df.head(10), x='Coefficient Magnitude', y='Feature', palette='viridis')
    plt.title('Top 10 Feature Coefficient Magnitudes')
    plt.show()

## Step 13: Summary of Insights & Conclusion
- **Month-to-month contracts** present the highest risk of churn.
- **Tenure** has an inverse relationship with churn; newer customers are highly unstable.
- **Hyperparameter Tuning** via GridSearchCV yielded a notable improvement in F1-score.
- **Gradient Descent** convergence demonstrates that a learning rate of `0.1` achieves stable minimization without divergence.